# Fakeddit dataset - Data Exploration Project

## The Start: Imports, Loadings, Libraries


In [ ]:
# Important downloads for on-machine use
%pip install polars
%pip install tqdm
%pip install transformers
%pip install -q -U accelerate bitsandbytes qwen-vl-utils
%pip install --force-reinstall torch torchvision
%pip install sentence-transformers scikit-learn
%pip install lightgbm
%pip install datasketch
%pip install optuna
%pip install pyarrow

# Important downloads for in-cloud use
# %pip install -q kaggle
# %mkdir -p /content/data/fakeddit
# %kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p /content/data/fakeddit --unzip
# %pip install -q -U accelerate bitsandbytes qwen-vl-utils

In [ ]:
# Basic imports
import polars as pl
import matplotlib
import plotly.express as px
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# Advanced imports
import transformers
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
print(transformers.__version__)
print("Qwen2.5-VL - OK")

In [ ]:
# all_data extraction in-cloud
# SAMPLES_DIR = Path("/content/data/fakeddit/multimodal_only_samples")

# all_data: pl.DataFrame = pl.concat([
#     pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
# ]).with_columns(
#     pl.from_epoch("created_utc", time_unit="s").alias("created_date")
# )

# all_data extraction on-machine
SAMPLES_DIR = Path("multimodal_only_samples") 
all_data: pl.DataFrame = pl.concat([pl.read_csv(filename, separator="\t") for filename in SAMPLES_DIR.iterdir() 
                                    if filename.is_file()]).with_columns(pl.from_epoch("created_utc", time_unit="s").alias("created_date"))

## Introductory EDA of Data

### Four basic methods for demostration and description

In [ ]:
all_data.shape

In [ ]:
all_data.schema

In [ ]:
all_data.head()

In [ ]:
all_data.describe()

Dataset consists of nearly 700k rows; each containing of 16 values.

### Main structure of label distribution

In the dataset each row of data is labeled with three types of labels:

**2-way label:** each post is classified as one containing true or false data.

**3-way labels:** posts containing false data have been split into two categories:
  - false text, false sample - sample has `False` label in 2-way labeling and contains untrue information;
  - true text, fale sample - sample has been deemed `False`, but contains "true" text - it involves only posts containing propaganda posters.

**6-way labels:** `True` label remains the same as in the other two kinds of labeling, the `False` label has been split into:
  - imposter content; i.e. content auto-generated by bots;
  - misleading content: content conveyed in a manipulated way to mislead people in believing false narratives;
  - satire/parody;
  - false connection, or the submission images in this category do not accurately support their text descriptions;
  - manipulated content, or the content that has been manually altered e.g. via image modifications.

In [ ]:
# 2-way labels
px.histogram(
    all_data.with_columns(
        label=pl.col("2_way_label").replace_strict({0: "False", 1: "True"}, return_dtype=pl.Utf8)
    ),
    x="label",
    title="Distribution of 2-way labels",
    histnorm="percent",
    text_auto=True,
).show()

In [ ]:
# 3-way labels
px.histogram(
    all_data.with_columns(
        label=pl.col("3_way_label")
        .replace_strict(
            {
                0: "True",
                1: "True text, False sample",
                2: "False text, False sample"
            }
            , return_dtype=pl.Utf8
        )
    ),
    x="label",
    title="Distribution of 3-way labels",
    histnorm="percent",
    text_auto=True,
).show()

In [ ]:
MAP_6_WAY = (
    {
        0: "True",
        1: "Satire / parody",
        2: "False connection",
        3: "Imposter content",
        4: "Manipulated",
        5: "Misleading",
    }
)

px.histogram(
    all_data.with_columns(
        label=pl.col("6_way_label")
        .replace_strict(MAP_6_WAY, return_dtype=pl.Utf8)
    ),
    x="label",
    title="Distribution of 6-way labels",
    histnorm="percent",
    text_auto=True,
).show()

### Further EDA int the context of the project

In [ ]:
titles_length = (
    all_data.select("clean_title")
    .drop_nulls()
    .with_columns(
        pl.col("clean_title")
        .str.split(" ")
        .list.len()
        .alias("word_count")
    )
)

titles_length.head()

In [ ]:
word_count_q99 = titles_length.select(pl.col("word_count").quantile(0.99)).item()

word_count_q99

In [ ]:
world_count_plot = px.histogram(
    titles_length.filter(pl.col('word_count') < word_count_q99),
    x="word_count",
    nbins=int(word_count_q99),
    title="Distribution of cleaned title word counts")

world_count_plot.update_layout(
    xaxis_title="Number of words in title",
    yaxis_title="Count")

world_count_plot.show()

In [ ]:
score_high = all_data.select(pl.col("score").quantile(0.95)).item()
score_low = all_data.select(pl.col("score").quantile(0.05)).item()

px.histogram(
    all_data.filter((score_low <= pl.col("score")) & (pl.col("score")  <= score_high)),
    x="score",
    nbins=100,
    title="Distribution of scores for subreddit posts")

In [ ]:
day_of_post = px.histogram(
    all_data,
    x="created_date",
    nbins=100,
    title="Post distribution over time")

day_of_post.show()

In [ ]:
six_labels_dist = px.histogram(
    all_data.with_columns(
        label=pl.col("6_way_label")
        .replace_strict(MAP_6_WAY, return_dtype=pl.Utf8)),
    x="subreddit",
    title="Distribution of 6-way labels",
    histnorm="percent",
    text_auto=True,
    color="label")

six_labels_dist.show()

**Fundamental conclusions**

The [paper](https://arxiv.org/pdf/1911.03854) that introduced the dataset has discussed the labeling strategy.
For each post in a given subreddit the same label was assigned, i.e. all `r/psbattle_artwork` are labeled as "Manipulated" - since it is a subreddit containing modified images (although they seem to serve satirical purpose rather than misleading one). Given high class imbalance and dubious labeling strategy, we've decided to choose 2-way labels for the classification experiments. We fear that for many cases intentions behind false information has been mislabeled. The 2-way labels - although not semantically rich - seem to be more reliable and trustworthy than 3 or 6-way ones.

## Cleaning The Data

For the purpose of making the text data easier to use for downstream tasks, we'll:
- get rid of duplicates (for now we'll test with exact duplicates)
- get rid of subreddit names; although it seems useful - it may introduce data leakage since there's always just one label for each samples associated with a particular subreddit.
- get rid of image urls since these won't be used, same goes `domain` column which identifies the image hosting service.
- handle `null` values in columns

In [ ]:
# Null values in a whole dataframe whole dataframe
nulls_in_data = (
    all_data.null_count()
    .transpose(include_header=True, column_names=["null_count"])
    .with_columns((pl.col("null_count") / all_data.height).alias("null_pct")))
with pl.Config(tbl_rows=-1, tbl_cols=-1): display(nulls_in_data) 

In [ ]:
messy_dubliKates = (
    all_data.group_by(["clean_title"])
      .len()
      .filter(pl.col("len") > 1)
      .sort("len", descending=True))

messy_dubliKates

In [ ]:
exact_dubliKates = (
    all_data.group_by(["clean_title", 'image_url'])
      .len()
      .filter(pl.col("len") > 1)
      .sort("len", descending=True)
)

exact_dubliKates

### Preparing Cleared Data

In [ ]:
trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")
trimmed_data.describe()

In [ ]:
display(all_data.filter(pl.col("num_comments").is_null()).head())
all_data.filter(pl.col("num_comments").is_null()).select(pl.col("subreddit").n_unique())

Missing values seem to correlate with `r/psbattle_artwork` subreddit. This one does not seem to exist anymore; it seems reasonable to remove it. We'll recreate `trimmed_data` in a messy way.

In [ ]:
trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")
trimmed_data.describe()

It did indeed fix the `null` situation.

## First Regression Models (The Baseline) 

This will be a simple, text-only TF-IDF model based on unigrams and bigrams. Metadata and images (additional modalities in p=other words) will be ommited for this step of project advancement.

In [ ]:
# Additional libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score)
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Self

In [ ]:
# Binary Baseline
# 0 (True) remains 0, the rest (1-5) become 1 (Fake)
model_data_bin = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
    .with_columns(
        pl.when(pl.col("6_way_label") == 0)
        .then(0)
        .otherwise(1)
        .alias("binary_label")
    )
)

X_bin = model_data_bin["clean_title"]
y_bin = model_data_bin["binary_label"]

# Train-test split
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

print("Class distribution (Training set - Binary):")
print(y_train_bin.value_counts(normalize=True))

# Defining and training the binary pipeline
pipeline_bin = Pipeline([
    ("TF-IDF", TfidfVectorizer(ngram_range=(1, 2), max_features=30_000, lowercase=True)),
    ("Logi", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipeline_bin.fit(X_train_bin, y_train_bin)

# Binary predictions
y_pred_bin = pipeline_bin.predict(X_test_bin)
y_proba_bin = pipeline_bin.predict_proba(X_test_bin)[:, 1] # Probability of class 1 (Fake)

# Binary evaluation metrics
roc_auc_bin = roc_auc_score(y_test_bin, y_proba_bin)
print(f"\nROC AUC (Binary): {roc_auc_bin:.4f}\n")
print(classification_report(y_test_bin, y_pred_bin, digits=4, target_names=["True (0)", "Fake (1)"]))

# Visualisation and interpretation

# 1. Confusion matrix visualization
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
px.imshow(
    cm_bin, text_auto=True, 
    x=["True (0)", "Fake (1)"], y=["True (0)", "Fake (1)"], 
    color_continuous_scale="Reds", 
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="Confusion Matrix - Binary Model"
).show()

# 2. Extracting the most important words for both binary classes
clf_bin = pipeline_bin.named_steps["Logi"]
vectorizer_bin = pipeline_bin.named_steps["TF-IDF"]
feature_names_bin = np.array(vectorizer_bin.get_feature_names_out())

# In a binary model, we only have one set of weights: coef_[0]
coefs_bin = clf_bin.coef_[0]

TOP_N = 10
top_true_idx = np.argsort(coefs_bin)[:TOP_N]           # Smallest (negative) weights -> True
top_fake_idx = np.argsort(coefs_bin)[-TOP_N:][::-1]    # Largest (positive) weights -> Fake

# Assembling the results into a readable table
top_features_df_bin = pd.DataFrame({
    "Words (True)": feature_names_bin[top_true_idx],
    "Weight (True)": coefs_bin[top_true_idx],
    "Words (Fake)": feature_names_bin[top_fake_idx],
    "Weight (Fake)": coefs_bin[top_fake_idx]
})
top_features_df_bin.index = [f"Rank {i+1}" for i in range(TOP_N)]

print("\nTop terms driving the binary predictions:")
display(top_features_df_bin)

In [ ]:
# 6-labels model
# Comparing text embeddings
model_data = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
)

display(model_data.shape)

X = model_data["clean_title"]
y = model_data["6_way_label"]

# Added "_multi" suffix so Stage 3 picks this up automatically
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train_multi))
print("Test size:", len(X_test_multi))
print("Train label distribution:")
print(y_train_multi.value_counts(normalize=True))

baseline_model = Pipeline([
    (
        "TF-IDF",
        TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=30_000,
            lowercase=True,
        ),
    ),
    (
        "Logi",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

baseline_model.fit(X_train_multi, y_train_multi)

y_pred = baseline_model.predict(X_test_multi)
y_proba = baseline_model.predict_proba(X_test_multi)

roc_auc = roc_auc_score(y_test_multi, y_proba, multi_class="ovr")

print(f"{roc_auc=}")
print(classification_report(y_test_multi, y_pred, digits=4))

# Dictionary with class names
label_names = {
    0: "True",
    1: "Satire / parody",
    2: "False connection",
    3: "Imposter content",
    4: "Manipulated",
    5: "Misleading",
}

clf = baseline_model.named_steps["Logi"]
vectorizer = baseline_model.named_steps["TF-IDF"]

class_labels = clf.classes_
string_labels = [label_names[c] for c in class_labels] # Generating text names

# Quick display of the confusion matrix
cm = confusion_matrix(y_test_multi, y_pred, labels=class_labels)
px.imshow(
    cm, 
    text_auto=True, 
    x=string_labels, 
    y=string_labels, 
    color_continuous_scale="Blues",
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="Confusion Matrix - Multiclass Baseline"
).show()

# Vectorized feature extraction
TOP_N = 20
feature_names = np.array(vectorizer.get_feature_names_out())

# Getting indices of TOP-N features for each class simultaneously (row - class, columns - indices)
top_indices = np.argsort(clf.coef_, axis=1)[:, -TOP_N:][:, ::-1]

# Creating a dictionary where the key is the class name, the value is the list of words
top_features_dict = {
    string_labels[i]: feature_names[top_indices[i]] 
    for i in range(len(string_labels))
}

# Conversion to a nice table for the report
top_features_df = pd.DataFrame(top_features_dict)
top_features_df.index = [f"Rank {i+1}" for i in range(TOP_N)]

print("Top terms driving the predictions for each class:")
display(top_features_df)

## Semantic Embeddings & Advanced Classifiers [time-critical on-machine; default test version]

While the baseline models relied on lexical frequency (TF-IDF), this stage leverages deep learning to capture the actual semantic meaning of the titles. 

We will use HuggingFace `SentenceTransformers` to convert text into dense vector representations (embeddings). To evaluate the best approach, we built an automated experimentation pipeline that tests multiple combinations of:
1. **Embedders:** `all-MiniLM-L12-v2` and `paraphrase-MiniLM-L3-v2` (Fast and efficient transformers).
2. **Classifiers:** Logistic Regression, Random Forest, and LightGBM.

This engine will train all combinations, evaluate their performance, log errors for further analysis, and save the artifacts.

In [ ]:
# Transformers for embeddings

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Self

class TitleEmbedder(TransformerMixin, BaseEstimator):
    """
    A custom scikit-learn transformer that wraps HuggingFace SentenceTransformers.
    This allows us to seamlessly integrate deep learning embeddings into standard ML pipelines.
    """
    def __init__(self, model_name: str, *, batch_size: int = 256) -> None:
        self._model_name = model_name
        self._batch_size = batch_size
        self._model = None # Initialized during fit

    def fit(self, X: np.ndarray, y: None = None) -> Self:
        # Load the pre-trained transformer model
        self._model = SentenceTransformer(self._model_name)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        # Convert text into dense vector embeddings
        return self._model.encode(
            list(X),
            batch_size=self._batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

In [ ]:
# Configuration setup of experiments.

import os
import gc
import torch
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# Define output directories for model artifacts.
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)
(artifacts_dir / "confusion_matrices").mkdir(exist_ok=True)
(artifacts_dir / "errors").mkdir(exist_ok=True)

# Re-using the labels from the baseline stage.
label_names = ["True", "Satire parody", "False connection", "Imposter content", "Manipulated", "Misleading"]

# Define the embedders to test.
embeddings_dict = {
    "minilm-really": ("emb", TitleEmbedder("all-MiniLM-L12-v2")),
    "paraphrase": ("emb", TitleEmbedder("paraphrase-MiniLM-L3-v2")),
}

# Define the classifiers to test.
classifiers_dict = {
    "logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    "lgbm": lambda: lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.1, class_weight="balanced", num_leaves=63, n_jobs=-1, random_state=42, verbose=-1
    ),
}

# Combine them into a grid of pipelines.
models_dict = {}
for emb_name, (step_name, emb_step) in embeddings_dict.items():
    for clf_name, clf_factory in classifiers_dict.items():
        model_key = f"{emb_name} and {clf_name}"
        models_dict[model_key] = Pipeline([
            (step_name, emb_step),
            ("clf", clf_factory()),
        ])

print(f"Total model configurations to train: {len(models_dict)}")
for k in models_dict:
    print(f"  {k}")

# Dataclass to store metrics for final comparison.
@dataclass
class ModelResult:
    name: str
    roc_auc: float
    f1_macro: float
    accuracy: float
    precision_macro: float
    recall_macro: float

In [ ]:
# Training and evaluation process.

import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Use background backend to avoid opening multiple blank windows in jupyter.
matplotlib.use("Agg")

results = []
all_errors = []

# Quick test mode. Remove for final run.
x_train_multi = X_train_multi[:500]
y_train_multi = y_train_multi[:500]
x_test_multi = X_test_multi[:100]
y_test_multi = y_test_multi[:100]

for model_name, model in tqdm(models_dict.items(), desc="Evaluating models"):
    print(f"\nTraining and evaluating {model_name}.")

    # Train the pipeline.
    model.fit(x_train_multi, y_train_multi)
    
    # Generate predictions.
    y_pred = model.predict(x_test_multi)
    y_proba = model.predict_proba(x_test_multi)

    # Calculate metrics.
    roc_auc = roc_auc_score(y_test_multi, y_proba, multi_class="ovr", average="macro")
    report = classification_report(y_test_multi, y_pred, target_names=label_names, digits=4, output_dict=True)

    result = ModelResult(
        name=model_name,
        roc_auc=roc_auc,
        f1_macro=report["macro avg"]["f1-score"],
        accuracy=report["accuracy"],
        precision_macro=report["macro avg"]["precision"],
        recall_macro=report["macro avg"]["recall"],
    )
    results.append(result)
    print(result)

    # Generate and save static confusion matrix.
    cm = confusion_matrix(y_test_multi, y_pred)
    fig_mpl, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    fig_mpl.colorbar(im, ax=ax)

    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(label_names, fontsize=8)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{model_name} roc auc: {roc_auc:.4f} f1 macro: {result.f1_macro:.4f}")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)

    fig_mpl.tight_layout()
    fig_mpl.savefig(artifacts_dir / "confusion_matrices" / f"{model_name}.png", dpi=200)
    plt.close(fig_mpl)

    # Show the interactive plotly version in the notebook.
    px.imshow(
        cm, x=label_names, y=label_names, text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predicted label", y="True label", color="Count"),
        title=f"{model_name} roc auc: {roc_auc:.4f} f1 macro: {result.f1_macro:.4f}",
        height=600,
    ).show()

    # Error analysis to save misclassified samples.
    y_test_np = np.array(y_test_multi)
    y_pred_np = np.array(y_pred)
    misclassified_mask = y_test_np != y_pred_np

    if misclassified_mask.any():
        x_test_list = list(x_test_multi)
        error_df = pd.DataFrame({
            "model": model_name,
            "index": np.where(misclassified_mask)[0],
            "text": [x_test_list[i] for i in np.where(misclassified_mask)[0]],
            "true_label": y_test_np[misclassified_mask],
            "true_label_name": [label_names[l] for l in y_test_np[misclassified_mask]],
            "pred_label": y_pred_np[misclassified_mask],
            "pred_label_name": [label_names[l] for l in y_pred_np[misclassified_mask]],
            "pred_proba_max": y_proba[misclassified_mask].max(axis=1),
        })
        error_df.to_csv(artifacts_dir / "errors" / f"{model_name} errors.tsv", sep="\t", index=False)
        all_errors.append(error_df)

    # Free memory.
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Aggregate and display final results.
results_df = pd.DataFrame([asdict(r) for r in results]).sort_values("f1_macro", ascending=False)
results_df.to_csv(artifacts_dir / "results_table.tsv", sep="\t", index=False)

if all_errors:
    combined_errors = pd.concat(all_errors, ignore_index=True)
    combined_errors.to_csv(artifacts_dir / "all_errors_combined.tsv", sep="\t", index=False)
    print(f"\nTotal misclassifications saved across all models: {len(combined_errors)}")

print(f"\nAll artifacts successfully saved to: {artifacts_dir.resolve()}")

print("\nFinal model comparison.")
display(results_df)

## Hyperparameters tuning with Optuna [time-critical on-machine; default test version]

In [ ]:
import optuna
import lightgbm as lgb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

# Set to False for the final full run.
use_quick_test = True

if use_quick_test:
    print("Running in quick test mode with reduced data and trials.")
    x_train_opt = X_train_multi[:500]
    y_train_opt = y_train_multi[:500]
    x_test_opt = X_test_multi[:100]
    y_test_opt = y_test_multi[:100]
    number_of_trials = 10
else:
    print("Running full optimization mode.")
    x_train_opt = X_train_multi
    y_train_opt = y_train_multi
    x_test_opt = X_test_multi
    y_test_opt = y_test_multi
    number_of_trials = 20

# Create an internal validation set for Optuna to avoid touching the final test set.
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_opt, y_train_opt, test_size=0.15, random_state=42, stratify=y_train_opt
)

print("Optimizing baseline model: TF-IDF and logistic regression.")

def objective_baseline(trial):
    # Optuna suggests hyperparameters for TF-IDF.
    max_df = trial.suggest_float('max_df', 0.7, 1.0)
    min_df = trial.suggest_int('min_df', 1, 5)
    
    # Optuna suggests hyperparameters for logistic regression.
    c_param = trial.suggest_float('c_param', 1e-3, 10.0, log=True)
    
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000, max_df=max_df, min_df=min_df, lowercase=True)),
        ("logreg", LogisticRegression(C=c_param, max_iter=1000, class_weight="balanced", random_state=42))
    ])
    
    pipeline.fit(x_tr, y_tr)
    y_pred_val = pipeline.predict(x_val)
    return f1_score(y_val, y_pred_val, average='macro')

study_baseline = optuna.create_study(direction='maximize')
study_baseline.optimize(objective_baseline, n_trials=number_of_trials)

print("Precomputing embeddings to save time during trials.")

best_embedder = TitleEmbedder("paraphrase-MiniLM-L3-v2", batch_size=256)
x_tr_emb = best_embedder.fit_transform(x_tr)
x_val_emb = best_embedder.transform(x_val)
x_test_emb = best_embedder.transform(x_test_opt)

print("Optimizing second model: paraphrase embeddings and logistic regression.")

def objective_emb_logreg(trial):
    c_param = trial.suggest_float('c_param', 1e-3, 10.0, log=True)
    model = LogisticRegression(C=c_param, max_iter=1000, class_weight="balanced", random_state=42)
    model.fit(x_tr_emb, y_tr)
    y_pred_val = model.predict(x_val_emb)
    return f1_score(y_val, y_pred_val, average='macro')

study_logreg = optuna.create_study(direction='maximize')
study_logreg.optimize(objective_emb_logreg, n_trials=number_of_trials)

print("Optimizing third model: paraphrase embeddings and LightGBM.")

def objective_emb_lgbm(trial):
    params = {
        'objective': 'multiclass', 'num_class': 6, 'class_weight': 'balanced',
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 80),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr_emb, y_tr,
        eval_set=[(x_val_emb, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    y_pred_val = model.predict(x_val_emb)
    return f1_score(y_val, y_pred_val, average='macro')

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_emb_lgbm, n_trials=number_of_trials)

print("Evaluating final results for all optimized models.")

final_results = []

# Evaluate best baseline model.
best_baseline_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000, 
                              max_df=study_baseline.best_params['max_df'], 
                              min_df=study_baseline.best_params['min_df'], lowercase=True)),
    ("logreg", LogisticRegression(C=study_baseline.best_params['c_param'], max_iter=1000, class_weight="balanced", random_state=42))
])

best_baseline_pipe.fit(x_train_opt, y_train_opt)
f1_base = f1_score(y_test_opt, best_baseline_pipe.predict(x_test_opt), average='macro')
final_results.append({"model": "TF-IDF and logistic regression", "optuna f1 score": f1_base})

# Evaluate best paraphrase and logistic regression model.
best_emb_logreg = LogisticRegression(C=study_logreg.best_params['c_param'], max_iter=1000, class_weight="balanced", random_state=42)
best_emb_logreg.fit(x_tr_emb, y_tr)
f1_emb_log = f1_score(y_test_opt, best_emb_logreg.predict(x_test_emb), average='macro')
final_results.append({"model": "Paraphrase and logistic regression", "optuna f1 score": f1_emb_log})

# Evaluate best paraphrase and LightGBM model.
lgbm_params = study_lgbm.best_params.copy()
lgbm_params.update({'objective': 'multiclass', 'num_class': 6, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbose': -1})
best_emb_lgbm = lgb.LGBMClassifier(**lgbm_params)
best_emb_lgbm.fit(x_tr_emb, y_tr)

y_test_pred_lgbm = best_emb_lgbm.predict(x_test_emb)
f1_emb_lgbm = f1_score(y_test_opt, y_test_pred_lgbm, average='macro')
final_results.append({"model": "Paraphrase and LightGBM", "optuna f1 score": f1_emb_lgbm})

final_df = pd.DataFrame(final_results).sort_values(by="optuna f1 score", ascending=False)
display(final_df)

In [ ]:
#Visualisations for reporting 
# try:
#     from IPython import get_ipython
#     get_ipython().run_line_magic('matplotlib', 'inline')
# except Exception:
#     pass
print("Generating visual outputs and reports for all optimized models.")

label_names = ["True", "Satire parody", "False connection", "Imposter content", "Manipulated", "Misleading"]

models_to_evaluate = {
    "Tf-idf and logistic regression": (best_baseline_pipe, x_test_opt),
    "Paraphrase and logistic regression": (best_emb_logreg, x_test_emb),
    "Paraphrase and lightgbm": (best_emb_lgbm, x_test_emb)
}

for model_name, (model_object, test_data) in models_to_evaluate.items():
    print(f"\nEvaluating performance for {model_name}.")
    
    predictions = model_object.predict(test_data)
    
    print("Classification report.")
    print(classification_report(y_test_opt, predictions, target_names=label_names, digits=4))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test_opt, predictions,
        display_labels=label_names,
        cmap='Blues',
        ax=ax,
        xticks_rotation=45
    )
    plt.title(f'Confusion matrix for {model_name}')
    plt.tight_layout()
    plt.show()

print("Generating feature importances for the best lightgbm model.")

importances = best_emb_lgbm.feature_importances_
feature_names_list = [f"Embedding dimension {i}" for i in range(x_tr_emb.shape[1])]

feature_importance_df = pd.DataFrame({
    'feature': feature_names_list, 
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feature_importance_df.head(20))
plt.title('Feature importances from lightgbm model')
plt.tight_layout()
plt.show()

## Multimodal Classification Using Vision-Language Models (VLMs) [time-critical on-machine; default test version]

In [ ]:
# DO NOT USE ON STANDARD LAPTOP! FOR THE FULL RUN ONLY!!!
import os
import time
import threading
import gc
from pathlib import Path
from hashlib import sha256
from io import BytesIO
from typing import Literal, TypedDict, Union
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import Counter

import requests
from requests.adapters import HTTPAdapter
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import numpy as np
import polars as pl
import pandas as pd
import torch
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

import asyncio
import random
import tarfile
from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlparse
import aiohttp

# Conservative download settings for the full run.
max_in_flight = 256
per_host_limit = 60
global_requests_per_second = 200
per_host_requests_per_second = 60
request_jitter_seconds = 0.05
write_workers = 16
write_queue_size = write_workers * 8
connect_timeout = 2
read_timeout = 2
max_retries = 3
base_backoff_seconds = 0.75
max_backoff_seconds = 8.0
backoff_factor = 2.0
retry_http_statuses = {408, 429, 500, 502, 503}

save_dir = Path("artifacts/images")
save_dir.mkdir(parents=True, exist_ok=True)
archive_path = Path("artifacts/images.tar.xz")

base_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:151.0) Gecko/20100101 Firefox/151.0"
user_agents = [f"{base_user_agent} test-{i}" for i in range(1, 100)]

class ImageOk(TypedDict):
    status: Literal["ok"]
    image_url: str
    image_path: str
    image_id: int
    user_agent: str

class ImageError(TypedDict):
    status: Literal["error"]
    image_url: str
    exception_type: str
    exception: str
    http_status: int | None
    user_agent: str | None

ImageResult = Union[ImageOk, ImageError]

class FetchOk(TypedDict):
    status: Literal["ok"]
    image_id: int
    image_url: str
    data: bytes
    content_type: str | None
    user_agent: str

FetchResult = Union[FetchOk, ImageError]

pil_format_to_ext = {
    "JPEG": ".jpg", "PNG": ".png", "WEBP": ".webp", "GIF": ".gif", "BMP": ".bmp", "AVIF": ".avif"
}

class AsyncRateLimiter:
    def __init__(self, requests_per_second: float, jitter_seconds: float = 0.0):
        self.min_interval = 1.0 / requests_per_second
        self.jitter_seconds = jitter_seconds
        self.lock = asyncio.Lock()
        self.last_request_at = 0.0

    async def wait(self) -> None:
        async with self.lock:
            loop = asyncio.get_running_loop()
            now = loop.time()
            wait_for = self.min_interval - (now - self.last_request_at)
            if wait_for > 0:
                await asyncio.sleep(wait_for)
            if self.jitter_seconds > 0:
                await asyncio.sleep(random.uniform(0, self.jitter_seconds))
            self.last_request_at = loop.time()

class PerHostRateLimiters:
    def __init__(self, requests_per_second: float, jitter_seconds: float = 0.0):
        self.requests_per_second = requests_per_second
        self.jitter_seconds = jitter_seconds
        self.limiters = {}
        self.lock = asyncio.Lock()

    async def wait(self, url: str) -> None:
        host = urlparse(url).netloc.lower() or "unknown-host"
        async with self.lock:
            limiter = self.limiters.get(host)
            if limiter is None:
                limiter = AsyncRateLimiter(requests_per_second=self.requests_per_second, jitter_seconds=self.jitter_seconds)
                self.limiters[host] = limiter
        await limiter.wait()

def validate_and_save_image_bytes(image_id: int, data: bytes) -> str:
    try:
        with Image.open(BytesIO(data)) as img:
            image_format = img.format
            img.load()
    except Exception as e:
        raise ValueError(f"Pil cannot load image: {e}") from e

    ext = pil_format_to_ext.get(image_format, ".img")
    path = save_dir / f"{image_id:012d}{ext}"
    path.write_bytes(data)
    return str(path)

def extension_from_url(url: str) -> str:
    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".avif"}:
        return suffix
    return ".img"

def extension_from_content_type(content_type: str) -> str:
    content_type = content_type.split(";", 1)[0].strip().lower()
    return {
        "image/jpeg": ".jpg", "image/png": ".png", "image/webp": ".webp", "image/gif": ".gif", "image/bmp": ".bmp", "image/avif": ".avif"
    }.get(content_type, ".img")

def exception_text(e: BaseException) -> str:
    return str(e) or repr(e)

def parse_retry_after_seconds(value: str | None) -> float | None:
    if not value:
        return None
    value = value.strip()
    if value.isdigit():
        return float(value)
    try:
        retry_time = parsedate_to_datetime(value)
        if retry_time.tzinfo is None:
            retry_time = retry_time.replace(tzinfo=timezone.utc)
        now = datetime.now(timezone.utc)
        return max(0.0, (retry_time - now).total_seconds())
    except Exception:
        return None

def backoff_seconds(attempt: int, retry_after: str | None = None) -> float:
    retry_after_seconds = parse_retry_after_seconds(retry_after)
    if retry_after_seconds is not None:
        return min(retry_after_seconds, max_backoff_seconds)
    deterministic = base_backoff_seconds * (backoff_factor ** attempt)
    jitter = random.uniform(0, request_jitter_seconds)
    return min(deterministic + jitter, max_backoff_seconds)

def choose_user_agent() -> str:
    return random.choice(user_agents)

def make_headers(user_agent: str) -> dict[str, str]:
    return {
        "User-Agent": user_agent,
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Connection": "keep-alive",
    }

async def fetch_image_bytes(session: aiohttp.ClientSession, image_id: int, url: str, global_rate_limiter: AsyncRateLimiter, per_host_rate_limiters: PerHostRateLimiters) -> FetchResult:
    last_exception_type = "UnknownError"
    last_exception = "Unknown error"
    last_http_status = None
    last_user_agent = None

    for attempt in range(max_retries + 1):
        user_agent = choose_user_agent()
        last_user_agent = user_agent
        try:
            await global_rate_limiter.wait()
            await per_host_rate_limiters.wait(url)
            async with session.get(url, headers=make_headers(user_agent)) as response:
                http_status = response.status
                last_http_status = http_status
                content_type = response.headers.get("Content-Type")

                if http_status >= 400:
                    response.release()
                    last_exception_type = "HTTPStatusError"
                    last_exception = f"Http status {http_status}"
                    if http_status in retry_http_statuses and attempt < max_retries:
                        sleep_for = backoff_seconds(attempt=attempt, retry_after=response.headers.get("Retry-After"))
                        await asyncio.sleep(sleep_for)
                        continue
                    return {"status": "error", "image_url": url, "exception_type": last_exception_type, "exception": last_exception, "http_status": http_status, "user_agent": user_agent}

                data = await response.read()
                return {"status": "ok", "image_id": image_id, "image_url": url, "data": data, "content_type": content_type, "user_agent": user_agent}

        except asyncio.TimeoutError as e:
            last_exception_type = "TimeoutError"
            last_exception = exception_text(e)
            last_http_status = None
            if attempt < max_retries:
                await asyncio.sleep(backoff_seconds(attempt))
                continue
        except aiohttp.ClientError as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None
            if attempt < max_retries:
                await asyncio.sleep(backoff_seconds(attempt))
                continue
        except Exception as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None
            break

    return {"status": "error", "image_url": url, "exception_type": last_exception_type, "exception": last_exception, "http_status": last_http_status, "user_agent": last_user_agent}

async def fetch_worker(url_queue: asyncio.Queue, write_queue: asyncio.Queue, session: aiohttp.ClientSession, global_rate_limiter: AsyncRateLimiter, per_host_rate_limiters: PerHostRateLimiters) -> None:
    while True:
        item = await url_queue.get()
        if item is None:
            url_queue.task_done()
            return
        image_id, url = item
        result = await fetch_image_bytes(session=session, image_id=image_id, url=url, global_rate_limiter=global_rate_limiter, per_host_rate_limiters=per_host_rate_limiters)
        await write_queue.put(result)
        url_queue.task_done()

async def write_worker(write_queue: asyncio.Queue, results_list: list, exception_counts: Counter, http_status_counts: Counter, user_agent_counts: Counter, pbar: tqdm) -> None:
    while True:
        result = await write_queue.get()
        if result is None:
            write_queue.task_done()
            return
        user_agent = result.get("user_agent")
        if user_agent:
            user_agent_counts[user_agent] += 1

        if result["status"] == "ok":
            try:
                image_path = await asyncio.to_thread(validate_and_save_image_bytes, result["image_id"], result["data"])
                results_list.append({"status": "ok", "image_url": result["image_url"], "image_path": image_path, "image_id": result["image_id"], "user_agent": result["user_agent"]})
            except Exception as e:
                exception_type = type(e).__name__
                exception_counts[exception_type] += 1
                http_status_counts[None] += 1
                results_list.append({"status": "error", "image_url": result["image_url"], "exception_type": exception_type, "exception": exception_text(e), "http_status": None, "user_agent": result["user_agent"]})
        else:
            exception_counts[result["exception_type"]] += 1
            http_status_counts[result["http_status"]] += 1
            results_list.append(result)

        pbar.update(1)
        write_queue.task_done()

def make_tar_xz(source_dir: Path, target_archive_path: Path) -> Path:
    target_archive_path.parent.mkdir(parents=True, exist_ok=True)
    if target_archive_path.exists():
        target_archive_path.unlink()
    with tarfile.open(target_archive_path, mode="w:xz") as tar:
        tar.add(source_dir, arcname=source_dir.name)
    print(f"Archive successfully created at {target_archive_path}.")
    return target_archive_path

async def fetch_all_images(urls: list[str]) -> list:
    results_list = []
    exception_counts = Counter()
    http_status_counts = Counter()
    user_agent_counts = Counter()

    timeout = aiohttp.ClientTimeout(total=None, connect=connect_timeout, sock_read=read_timeout)
    connector = aiohttp.TCPConnector(limit=max_in_flight, limit_per_host=per_host_limit, ttl_dns_cache=300, enable_cleanup_closed=True)
    global_rate_limiter = AsyncRateLimiter(requests_per_second=global_requests_per_second, jitter_seconds=request_jitter_seconds)
    per_host_rate_limiters = PerHostRateLimiters(requests_per_second=per_host_requests_per_second, jitter_seconds=request_jitter_seconds)

    url_queue = asyncio.Queue(maxsize=max_in_flight * 2)
    write_queue = asyncio.Queue(maxsize=write_queue_size)

    start = time.time()

    async with aiohttp.ClientSession(connector=connector, timeout=timeout, raise_for_status=False) as session:
        with tqdm(total=len(urls), desc="Downloading images asynchronously") as pbar:
            fetch_tasks = [asyncio.create_task(fetch_worker(url_queue=url_queue, write_queue=write_queue, session=session, global_rate_limiter=global_rate_limiter, per_host_rate_limiters=per_host_rate_limiters)) for _ in range(max_in_flight)]
            write_tasks = [asyncio.create_task(write_worker(write_queue=write_queue, results_list=results_list, exception_counts=exception_counts, http_status_counts=http_status_counts, user_agent_counts=user_agent_counts, pbar=pbar)) for _ in range(write_workers)]

            for image_id, url in enumerate(urls):
                await url_queue.put((image_id, url))

            for _ in fetch_tasks:
                await url_queue.put(None)

            await url_queue.join()
            await asyncio.gather(*fetch_tasks)

            for _ in write_tasks:
                await write_queue.put(None)

            await write_queue.join()
            await asyncio.gather(*write_tasks)

    elapsed = time.time() - start
    
    print("Creating tar archive for cloud storage.")
    make_tar_xz(source_dir=save_dir, target_archive_path=archive_path)

    ok_count = sum(1 for row in results_list if row["status"] == "ok")
    error_count = sum(1 for row in results_list if row["status"] == "error")

    print(f"Downloaded: {ok_count}")
    print(f"Errors: {error_count}")
    print(f"Elapsed: {elapsed:.2f} seconds.")

    return results_list

# Execute the new asynchronous downloader.
print("Preparing to download the dataset.")
urls_to_download = all_data.select("image_url").drop_nulls().unique().get_column("image_url").to_list()

# Run the async function natively in jupyter.
results = await fetch_all_images(urls_to_download)

In [ ]:
# DO NOT USE ON STANDARD LAPTOP! FOR THE FULL RUN ONLY!!!
# Preparing the combined multimodal dataset.
manifest = pl.DataFrame(results)
model_data = (
    all_data.unique(subset=["clean_title", "image_url"], keep="first")
    .join(manifest.filter(pl.col("status") == "ok").select(["image_url", "image_path"]), on="image_url", how="inner")
    .select(["clean_title", "image_path", "6_way_label"])
    .drop_nulls()
).to_pandas()

x_train, x_test, y_train, y_test = train_test_split(
    model_data[["clean_title", "image_path"]], model_data["6_way_label"].astype(int),
    test_size=0.2, random_state=42, stratify=model_data["6_way_label"]
)

# Initializing the large vision language model with 4-bit quantization for GPU memory efficiency.
print("Loading the massive Qwen 2.5 (7B parameters) vision model into GPU memory.")
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
vision_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16, quantization_config=quantization_config)
vision_model.eval()

@torch.inference_mode()
def encode_dataframe(df_part, batch_size=8, out_path=None):
    embeddings = []
    titles = df_part["clean_title"].astype(str).tolist()
    paths = df_part["image_path"].astype(str).tolist()
    
    for i in tqdm(range(0, len(df_part), batch_size), desc="Generating multimodal embeddings on GPU"):
        batch_titles = titles[i:i + batch_size]
        batch_paths = paths[i:i + batch_size]
        
        messages = [[{"role": "user", "content": [{"type": "image", "image": p}, {"type": "text", "text": f"Represent this Reddit post for classification.\nTitle: {t}"}]}] for t, p in zip(batch_titles, batch_paths)]
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(vision_model.device)
        outputs = vision_model(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
        
        hidden_state = outputs.hidden_states[-1]
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden_state.dtype)
        emb = (hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embeddings.append(emb.float().cpu().numpy())
        
        del inputs, outputs, hidden_state, mask
        gc.collect()
        torch.cuda.empty_cache()
        
    final_embeddings = np.vstack(embeddings).astype(np.float32)
    if out_path:
        np.save(out_path, final_embeddings)
    return final_embeddings

# Generating embeddings.
print("Extracting training and testing features.")
x_train_emb = encode_dataframe(x_train, batch_size=16, out_path="artifacts/qwen25vl_train.npy")
x_test_emb = encode_dataframe(x_test, batch_size=16, out_path="artifacts/qwen25vl_test.npy")

# Training the final classifier.
print("Training final LightGBM classifier on multimodal embeddings.")
classifier = lgb.LGBMClassifier(
    objective="multiclass", num_class=6, n_estimators=800, learning_rate=0.03, 
    num_leaves=63, subsample=0.9, colsample_bytree=0.9, class_weight="balanced", 
    n_jobs=-1, random_state=42, verbose=-1
)
classifier.fit(x_train_emb, y_train, eval_set=[(x_test_emb, y_test)], eval_metric="multi_logloss", callbacks=[lgb.early_stopping(5)])

predicted_probabilities = classifier.predict_proba(x_test_emb)
predicted_classes = predicted_probabilities.argmax(axis=1)

# Ensuring matplotlib works inline.
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Final evaluation with safety checks for flexible dataset sizes.
print("\nFinal multimodal performance metrics.")
print(f"Accuracy: {accuracy_score(y_test, predicted_classes):.4f}")
print(f"F1 macro: {f1_score(y_test, predicted_classes, average='macro'):.4f}")

# Roc-auc might fail if the model didn't see all classes during training (mostly an issue for tiny tests).
try:
    auc = roc_auc_score(y_test, predicted_probabilities, multi_class='ovr', average='macro')
    print(f"ROC-AUC macro: {auc:.4f}")
except ValueError:
    print("ROC-AUC macro: Not available due to missing classes in this batch.")

label_names = ["True", "Satire / parody", "False connection", "Imposter content", "Manipulated", "Misleading"]
print("\nClassification report.")

# Dynamically map only the classes the model actually learned.
seen_labels = classifier.classes_
seen_names = [label_names[i] for i in seen_labels]

print(classification_report(y_test, predicted_classes, labels=seen_labels, target_names=seen_names, digits=4, zero_division=0))

# Visual output for confusion matrix.
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, predicted_classes,
    labels=seen_labels,
    display_labels=seen_names,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45
)
plt.title('Confusion matrix for multimodal model')
plt.tight_layout()
plt.show()

# Visual output for feature importances.
print("Generating feature importances chart.")
importances = classifier.feature_importances_
feature_names_list = [f"Embedding dimension {i}" for i in range(x_train_emb.shape[1])]

feature_importance_df = pd.DataFrame({
    'feature': feature_names_list, 
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feature_importance_df.head(20))
plt.title('Feature importances from multimodal LightGBM')
plt.tight_layout()
plt.show()

In [ ]:
# Test and try for standard laptops

import os
import time
import threading
import gc
from pathlib import Path
from hashlib import sha256
from io import BytesIO
from typing import Literal, TypedDict, Union
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import Counter

import requests
from requests.adapters import HTTPAdapter
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import numpy as np
import polars as pl
import pandas as pd
import torch
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# Configuration parameters adapted for a standard laptop
max_workers = 16
max_in_flight = 64
timeout_seconds = 5
jpeg_quality = 85
max_long_side = 512
image_dir = Path("artifacts/images")
image_dir.mkdir(parents=True, exist_ok=True)
thread_local_storage = threading.local()

class ImageOk(TypedDict):
    status: Literal["ok"]
    image_url: str
    image_path: str

class ImageError(TypedDict):
    status: Literal["error"]
    image_url: str
    exception_type: str
    exception: str
    http_status: int | None

def get_session() -> requests.Session:
    if not hasattr(thread_local_storage, "session"):
        session = requests.Session()
        session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"})
        adapter = HTTPAdapter(pool_connections=max_workers, pool_maxsize=max_workers, max_retries=0)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        thread_local_storage.session = session
    return thread_local_storage.session

def fetch_and_save_image(url: str) -> Union[ImageOk, ImageError]:
    hashed_name = sha256(url.encode("UTF-8")).hexdigest() + ".jpg"
    path = image_dir / hashed_name
    
    if path.exists():
        return {"status": "ok", "image_url": url, "image_path": str(path)}
        
    tmp_path = path.with_suffix(".jpg.tmp")
    
    try:
        response = get_session().get(url, timeout=timeout_seconds)
        if response.status_code != 200:
            return {"status": "error", "image_url": url, "exception_type": "HTTPError", "exception": f"Http status {response.status_code}", "http_status": response.status_code}
            
        img = Image.open(BytesIO(response.content))
        img = ImageOps.exif_transpose(img).convert("RGB")
        img.thumbnail((max_long_side, max_long_side), Image.Resampling.LANCZOS)
        img.save(tmp_path, format="JPEG", quality=jpeg_quality, subsampling=2, optimize=True)
        tmp_path.replace(path)
        return {"status": "ok", "image_url": url, "image_path": str(path)}
        
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)
        return {"status": "error", "image_url": url, "exception_type": type(e).__name__, "exception": repr(e), "http_status": None}

# Downloading process limited to 20 images for cpu testing.
sample_limit = 20
print(f"Preparing to download up to {sample_limit} images for local testing.")

urls = all_data.select("image_url").drop_nulls().unique().get_column("image_url").to_list()[:sample_limit]
results = []
exception_counts = Counter()
urls_iter = iter(urls)

start_time = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    in_flight = set()
    for _ in range(min(max_in_flight, len(urls))):
        try:
            in_flight.add(executor.submit(fetch_and_save_image, next(urls_iter)))
        except StopIteration:
            break
            
    with tqdm(total=len(urls), desc="Downloading images") as pbar:
        while in_flight:
            done, in_flight = wait(in_flight, return_when=FIRST_COMPLETED)
            for future in done:
                result = future.result()
                results.append(result)
                if result["status"] == "error":
                    exception_counts[result["exception_type"]] += 1
                pbar.update(1)
                try:
                    in_flight.add(executor.submit(fetch_and_save_image, next(urls_iter)))
                except StopIteration:
                    pass

elapsed_time = time.time() - start_time
print(f"Download complete. {len(results)} images processed in {elapsed_time:.1f} seconds.")

# Preparing the combined multimodal dataset.
manifest = pl.DataFrame(results)
model_data = (
    all_data.unique(subset=["clean_title", "image_url"], keep="first")
    .join(manifest.filter(pl.col("status") == "ok").select(["image_url", "image_path"]), on="image_url", how="inner")
    .select(["clean_title", "image_path", "6_way_label"])
    .drop_nulls()
).to_pandas()

x_train, x_test, y_train, y_test = train_test_split(
    model_data[["clean_title", "image_path"]], model_data["6_way_label"].astype(int),
    test_size=0.2, random_state=42, 
    # stratify=model_data["6_way_label"] uncomment to make more valuable test
)

# Initializing the lightweight vision model for cpu.
print("Loading the lightweight Qwen 2.5 model (3B parameters) onto the cpu.")
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)

# Loading directly to cpu without bitsandbytes quantization.
vision_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name, 
    device_map="cpu", 
    torch_dtype=torch.float32
)
vision_model.eval()

@torch.inference_mode()
def encode_dataframe(df_part, batch_size=1, out_path=None):
    embeddings = []
    titles = df_part["clean_title"].astype(str).tolist()
    paths = df_part["image_path"].astype(str).tolist()
    
    for i in tqdm(range(0, len(df_part), batch_size), desc="Generating embeddings on cpu"):
        batch_titles = titles[i:i + batch_size]
        batch_paths = paths[i:i + batch_size]
        
        messages = [[{"role": "user", "content": [{"type": "image", "image": p}, {"type": "text", "text": f"Represent this Reddit post for classification.\nTitle: {t}"}]}] for t, p in zip(batch_titles, batch_paths)]
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(vision_model.device)
        outputs = vision_model(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
        
        hidden_state = outputs.hidden_states[-1]
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden_state.dtype)
        emb = (hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embeddings.append(emb.float().cpu().numpy())
        
        del inputs, outputs, hidden_state, mask
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    final_embeddings = np.vstack(embeddings).astype(np.float32)
    if out_path:
        np.save(out_path, final_embeddings)
    return final_embeddings

# Generating embeddings with batch size 1 to prevent cpu ram overflow.
print("Extracting features. This will take some time on a cpu.")
x_train_emb = encode_dataframe(x_train, batch_size=1, out_path="artifacts/qwen2vl_train.npy")
x_test_emb = encode_dataframe(x_test, batch_size=1, out_path="artifacts/qwen2vl_test.npy")

# Training the final classifier.
print("Training final LightGBM classifier on multimodal embeddings.")
classifier = lgb.LGBMClassifier(
    objective="multiclass", num_class=6, n_estimators=800, learning_rate=0.03, 
    num_leaves=63, subsample=0.9, colsample_bytree=0.9, class_weight="balanced", 
    n_jobs=-1, random_state=42, verbose=-1
)
classifier.fit(x_train_emb, y_train, eval_set=[(x_test_emb, y_test)], eval_metric="multi_logloss", callbacks=[lgb.early_stopping(5)])

predicted_probabilities = classifier.predict_proba(x_test_emb)
predicted_classes = predicted_probabilities.argmax(axis=1)

# Ensuring matplotlib works inline.
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Final evaluation with safety checks for small datasets.
print("\nFinal multimodal performance metrics.")
print(f"Accuracy: {accuracy_score(y_test, predicted_classes):.4f}")
print(f"F1 macro: {f1_score(y_test, predicted_classes, average='macro'):.4f}")

# Roc-auc might fail if the model didn't see all classes during training.
try:
    auc = roc_auc_score(y_test, predicted_probabilities, multi_class='ovr', average='macro')
    print(f"ROC-AUC macro: {auc:.4f}")
except ValueError:
    print("ROC-AUC macro: Not available for this small test batch (missing classes).")

label_names = ["True", "Satire / parody", "False connection", "Imposter content", "Manipulated", "Misleading"]
print("\nClassification report.")

# Dynamically map only the classes the model actually learned during this small test.
seen_labels = classifier.classes_
seen_names = [label_names[i] for i in seen_labels]

print(classification_report(y_test, predicted_classes, labels=seen_labels, target_names=seen_names, digits=4, zero_division=0))

# Visual output for confusion matrix.
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, predicted_classes,
    labels=seen_labels,
    display_labels=seen_names,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45
)
plt.title('Confusion matrix for multimodal model')
plt.tight_layout()
plt.show()

# Visual output for feature importances.
print("Generating feature importances chart.")
importances = classifier.feature_importances_
feature_names_list = [f"Embedding dimension {i}" for i in range(x_train_emb.shape[1])]

feature_importance_df = pd.DataFrame({
    'feature': feature_names_list, 
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feature_importance_df.head(20))
plt.title('Feature importances from multimodal LightGBM')
plt.tight_layout()
plt.show()